In [2]:
import torch 
import torch.nn as nn
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision import transforms
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(42)

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], 
                             [0.2470, 0.2435, 0.2616])
])
train_set = datasets.CIFAR10(
    download = True,
    root = 'data',
    train = True,
    transform = transform
)
test_set = datasets.CIFAR10(
    download = True,
    train = False,
    root = 'data',
    transform = transform
)
trainloader = DataLoader(
    train_set, 
    shuffle = True,
    batch_size = 256,
    num_workers = 4
)
testloader = DataLoader(
    test_set,
    shuffle = True,
    batch_size = 256,
    num_workers = 4
)


In [4]:
class CNNModel(nn.Module):
    def __init__(self, num_classes = 10, dropout_rate = 0.3):
        super(CNNModel, self).__init__()
        self.conv_layer1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(32)
        )
        self.conv_layer2 = nn.Sequential(
            nn.Conv2d(32, 32, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2),
            nn.Dropout(dropout_rate)
        )
        self.conv_layer3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(64)
        )
        self.conv_layer4 = nn.Sequential(
            nn.Conv2d(64, 64, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2),
            nn.Dropout(dropout_rate)
        )
        self.conv_layer5 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(128)
        )
        self.conv_layer6 = nn.Sequential(
            nn.Conv2d(128, 128, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(128)
        )
        self.conv_layer7 = nn.Sequential(
            nn.Conv2d(128, 128, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2),
            nn.Dropout(dropout_rate)
        )
        self.conv_layer8 = nn.Sequential(
            nn.Conv2d(128, 256, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(256)
        )
        self.conv_layer9 = nn.Sequential(
            nn.Conv2d(256, 256, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(256)
        )
        self.conv_layer10 = nn.Sequential(
            nn.Conv2d(256, 256, 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2, 2),
            nn.Dropout(dropout_rate)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 2 * 2, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )
        self._init_weights()
    def forward(self, x):
        x = self.conv_layer1(x)
        x = self.conv_layer2(x)
        x = self.conv_layer3(x)
        x = self.conv_layer4(x)
        x = self.conv_layer5(x)
        x = self.conv_layer6(x)
        x = self.conv_layer7(x)
        x = self.conv_layer8(x)
        x = self.conv_layer9(x)
        x = self.conv_layer10(x)
        out = self.classifier(x)
        return out
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode = 'fan_in', nonlinearity = 'relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    

In [5]:
model = CNNModel()
model.to(device)
print(model)

CNNModel(
  (conv_layer1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
  (conv_layer2): Sequential(
    (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.3, inplace=False)
  )
  (conv_layer3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
  (conv_layer4): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, tr

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3, weight_decay = 5e-4)

In [ ]:
epochs = 100
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [8]:
def evaluate(model, criterion, testloader):
    model.eval()
    test_loss = 0.0
    test_accuracy = 0
    total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * labels.size(0)

            predicted = torch.argmax(outputs.data, dim = 1)

            test_accuracy += (predicted == labels).sum().item()

            total += labels.size(0)
    test_loss /= total
    test_accuracy = test_accuracy * 100 / total
    return test_loss, test_accuracy



In [ ]:
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_accuracy = 0
    total = 0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        train_loss += loss.item() * labels.size(0)

        loss.backward()

        predicted = torch.argmax(outputs.data, dim = 1)

        train_accuracy += (predicted == labels).sum().item()

        total += labels.size(0)

        optimizer.step()
    train_loss /= total
    train_accuracy = train_accuracy * 100 / total

    test_loss, test_accuracy = evaluate(model, criterion, testloader)

    print(
        f'Train Loss: {train_loss:.4f}',
        f'Test Loss: {test_loss:.4f}',
        f'Train Accuracy: {train_accuracy:.2f}%',
        f'Test Accuracy: {test_accuracy:.2f}% '
    )
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)
